# Breach clocks: CERT-In 6 hours, DPDP, RBI and GDPR

Rehearse a personal-data breach at a microfinance company after India's DPDP breach rules take effect: start the clock, scope the breach from the logs, file with CERT-In inside six hours while injects arrive, work out every other clock that applies (RBI, DPDP, GDPR), and write the Board note, the notice to customers and the notification order.

The files for this seminar are in `~/case` (Linux) and `case/` (Python): `affected_customers.csv`, `api_access.log`, `certin-kaveri.json`, `README.txt`, `tabletop-kaveri-clocks.json`, `waf_alert.txt`.

## 1. Start the clock

Every breach clock starts at the moment **you noticed**, not when the attack began and not when you finished investigating. Read the analyst's note and the README. Write down the detection time in UTC, and work out when the six-hour reports are due.

- [ ] When did the attack start? When did Kaveri notice?
- [ ] Which time starts the clocks?
- [ ] When are the six-hour reports due, in UTC and in IST?

> Hint: The analyst reviewed the WAF digest at 10:20 IST. IST is UTC+5:30.

?> At what time (UTC, HH:MM) did Kaveri notice the breach? :: sha256:dd0342a8a310f5d3335bb94c975948cbb5d7a3079a769d03b218a588137f8192
?> By what time (UTC, HH:MM) must CERT-In be told? :: sha256:0f654d53ec914752804c4f1b88951a20ee1dd92b74c7b8ca518764841a106429

In [ ]:
%%bash
cd ~/case
cat README.txt
echo ---
cat waf_alert.txt

## 2. Scope it from the logs

Regulators will ask three questions first: what happened, how much, and to whom. Answer the first two from the API log. The log's times are in IST, so convert them before you write anything down.

- [ ] How many requests came from 198.51.100.66? How many different customer IDs?
- [ ] When was the first and the last one (IST, then UTC)?
- [ ] What does a normal app request look like, for comparison?

> Hint: `grep -o '/customers/[0-9]*' | sort -u | wc -l` counts different IDs. `head -1` and `tail -1` give the first and last line.

?> How many different customer records did 198.51.100.66 receive? :: sha256:28e7234668777f9ed7a63b82eac501322fa9ac707238d8a3e9e89c599458ab13
?> At what time (UTC, HH:MM) did the scraping start? :: sha256:d93906bfee503ab3c549c2467d06edf18c3d2f6e6f7dd665fa0e13b0c46a21b7

In [ ]:
%%bash
cd ~/case
grep -c '^198.51.100.66 ' api_access.log
grep '^198.51.100.66 ' api_access.log | grep -o '/customers/[0-9]*' | sort -u | wc -l
echo ---
grep '^198.51.100.66 ' api_access.log | head -1
grep '^198.51.100.66 ' api_access.log | tail -1
echo ---
grep -v '^198.51.100.66 ' api_access.log | head -3

## 3. Who is affected

The third question: to whom. The response log lists every record the scraper received and the country each customer lives in. The countries decide which clocks apply: everyone is covered by India's rules, and residents of the EU may bring in GDPR.

- [ ] Count the records by country
- [ ] Which fields were exposed? Which are the most sensitive?
- [ ] How many affected customers live in the EU?

> Hint: `cut -d, -f2 | sort | uniq -c` counts by the second column. Skip the header line with `tail -n +2`. Germany is DE, Ireland IE; the UAE (AE) is not in the EU.

?> How many affected customers live in the EU? :: sha256:a68b412c4282555f15546cf6e1fc42893b7e07f271557ceb021821098dd66c1b

In [ ]:
%%bash
cd ~/case
tail -n +2 affected_customers.csv | cut -d, -f2 | sort | uniq -c | sort -rn
echo ---
head -2 affected_customers.csv

## 4. File with CERT-In inside six hours

File the CERT-In report now, with what you know. The form checks your times against the evidence and says what's missing. After you file, advance the clock: replies and new facts arrive in the inbox. Answer them, and send an update when the facts change.

```tool certin
{"file": "certin-kaveri.json", "form": {"name": "Meenal Rao", "role": "Chief Information Security Officer", "email": "ciso@kaveri.example", "phone": "+91 80 4000 1200", "org": "Kaveri Microfinance Ltd", "sector": "Finance (NBFC regulated by RBI)", "occurred": "2027-06-08 03:10", "detected": "2027-06-08 04:50", "type": "Data breach", "systems": "Loan app API api.kaveri.example (/v2/customers/{id}/kyc), Linux servers behind the WAF; customer KYC database.", "symptoms": "One client with a valid session for customer 118842 requested other customers' KYC records in sequence from 03:10 to 04:18 UTC. The API returned 2,060 records: name, mobile, loan amount, last four digits of Aadhaar, address. A missing ownership check, released on 21 May 2027.", "indicators": "198.51.100.66; User-Agent python-requests/2.31.0; sequential requests to /v2/customers/{id}/kyc with the token of customer 118842.", "actions": "Blocked 198.51.100.66 at the WAF; switched rule 9021 to block; revoked the token; ownership check fix deployed 07:15 UTC; logs preserved; DPO and Board informed.", "personal": "yes", "filed": "2027-06-08 08:15"}, "n": 0, "blank": ["form"]}
```

- [ ] Detection time matches the evidence
- [ ] The incident type fits; personal data marked
- [ ] Indicators, affected systems and actions so far
- [ ] Filed before 10:50 UTC
- [ ] Inbox answered; an update sent after the CISO's message

> Hint: "Data breach" fits best: personal data was taken by someone who shouldn't have it. Partial facts are fine; say what you don't know yet in the symptoms.

?> Which CERT-In incident type fits best? :: sha256:e65afef11689fa692e80268c602646c3938ed3bbd838b7d8f2ebcc19bf90e7a2

## 5. Every clock, in order

CERT-In is only the first clock. Open the **Notification clocks** panel in the CERT-In tool: it shows which clocks apply to this case and when each is due. Turn it into a one-page **notification order**: who is told, by when, by whom, and the status.

| # | Who | Rule (from the tool's summary) | Due (UTC) | Owner | Status |
|---|---|---|---|---|---|
| 1 | CERT-In | Six hours from noticing (Directions of 28 April 2022) | 8 Jun 10:50 | CISO | Filed 08:15, update 09:35 |
| 2 | RBI | Six hours from detection for regulated entities | 8 Jun 10:50 | Head of compliance | Filed 08:30 |
| 3 | Data Protection Board, first notice | Without delay | 8 Jun, today | DPO | Sent 09:10 |
| 4 | Each affected customer | Without delay, plain language | 8 Jun, today | DPO with customer care | SMS and email from 10:00 |
| 5 | EU lead authority (Ireland) | 72 hours from awareness, if GDPR applies | 11 Jun 04:50 | DPO with EU counsel | Drafting |
| 6 | Data Protection Board, detailed report | Within 72 hours | 11 Jun 04:50 | DPO | Drafting |

- [ ] Every clock that applies is in the table, none that don't
- [ ] One owner for each
- [ ] Due times in UTC
- [ ] Which two are due first?

> Hint: The DPDP Rules ask for the Board and each affected person to be told without delay, then a detailed report to the Board within 72 hours. GDPR's 72 hours runs from becoming aware.

?> How many hours after noticing is the detailed report to the Data Protection Board due? :: sha256:8722616204217eddb39e7df969e0698aed8e599ba62ed2de1ce49b03ade0fede

## 6. Decide with four regulators watching

Play the tabletop as the CISO. The pressure is real: a CEO who wants to wait, a DPO who wants a plan, a journalist with a sample.

```tool tabletop
{"file": "tabletop-kaveri-clocks.json", "path": [{"node": "k1", "c": 0, "label": "File CERT-In and RBI with what we know now, and update later"}, {"node": "k2", "c": 0, "label": "Tell each affected customer without delay, in plain words, by SMS and email, with what they can do"}, {"node": "k3", "c": 0, "label": "Confirm the incident briefly, say customers are being told, give no data"}, {"node": "k4", "c": 0, "label": "Check whether GDPR applies to Kaveri Remit, and if so notify the lead EU authority within 72 hours"}], "n": 8, "blank": ["path"]}
```

- [ ] Play it as the CISO, then as the CEO
- [ ] One line for each decision: why

?> What is the best score you can get in this tabletop? :: sha256:2c624232cdd221771294dfbb310aca000a0df6ac8b66b696d90ef06fdefb64a3

## 7. The DPDP pack: the Board note and the notice

Two more readers. The **Board** needs to decide; the **customers** need to know what happened to them and what to do. The notice must be short and plain, in the language the customer uses with Kaveri, and say five things: what happened (nature, extent, timing), the likely consequences, what Kaveri has done, what the person can do, and who to contact.

```tool report
{"tpl": "escalation", "title": "Note to the Board: personal data breach in the loan app", "text": "# Note to the Board: personal data breach in the loan app\n\n## The ask\n\nPlease note this breach today and approve: free fraud-alert SMS for the 2,060 customers for 12 months, and an external review of the app's access checks before the next release.\n\n## Severity and why\n\nHigh. 2,060 customers' names, mobile numbers, addresses, loan amounts and the last four digits of their Aadhaar were taken. That is enough for convincing fraud calls.\n\n## What we know\n\n- On 8 June 2027, 03:10 to 04:18 UTC, one client (198.51.100.66) using one customer's session read 2,060 other customers' KYC records through the app's API.\n- The cause: a missing ownership check, released on 21 May. We searched back to then and found no other client doing this.\n- The fix went live at 07:15 UTC; the address is blocked and the token revoked.\n\n## What we don't know\n\n- Who is behind 198.51.100.66, and whether the records have been shared. A journalist says she has a sample.\n\n## Impact so far\n\n2,060 customers in India (1,932), the UAE (64), Germany (41) and Ireland (23). No money was moved and no passwords were exposed.\n\n## What I've done\n\nCERT-In filed 08:15 UTC and updated 09:35; RBI filed 08:30; the Data Protection Board told at 09:10; customers are being told by SMS and email from 10:00. The detailed report to the Board is due by 11 June 04:50 UTC.\n\n## Evidence\n\napi_access.log, affected_customers.csv and the WAF alert, preserved with hashes; the CERT-In reference is in the filing.\n", "n": 7, "blank": ["text"]}
```

Write the notice to customers in a Notes cell. A model:

<details><summary>Model notice to customers</summary>

Subject: We are writing to tell you about a problem with your Kaveri account details

On 8 June 2027, between 8:40 and 9:48 in the morning, someone used a fault in the Kaveri app to see some customers' details. Your details were among them.

What they saw: your name, mobile number, address, loan amount and the last four digits of your Aadhaar number. They did not see your password, your full Aadhaar number or your bank details, and no money was taken from any account.

What could happen: you may get calls or messages from people pretending to be Kaveri or a bank, who already know some of your details.

What we have done: we fixed the fault the same morning, blocked the person responsible, and reported it to CERT-In, the Reserve Bank of India and the Data Protection Board.

What you can do: Kaveri will never ask for your OTP, PIN or password by phone or message. If someone does, hang up and call us. We are sending you free fraud-alert messages for 12 months.

Questions: call 1800 000 4321 (free, 8 am to 8 pm) or write to dpo@kaveri.example. Arvind Shetty, Data Protection Officer, will answer.

</details>

- [ ] Board note: the ask first, then the facts
- [ ] Notice: all five parts, no jargon, no "we take your privacy seriously"
- [ ] Notice: what Kaveri will never ask for
- [ ] A contact who can actually answer

> Hint: Read your notice aloud. If your grandmother couldn't act on it, rewrite it.

?> How many affected customers live in India? :: sha256:01a0123885ebec5b37b52ddc058c20d052525c654b69e7b7bfd5feb291428bba

## 8. Hand in

Hand in this notebook. It carries every answer, tool result and report from the steps above.

**Rubric**

| Criterion | Points |
|---|---|
| Clock started at the right moment, with the evidence for it | 3 |
| Scope measured from the logs: requests, records and countries | 4 |
| CERT-In report filed within six hours: right type, indicators, systems and actions | 5 |
| Inbox answered and an update sent with the new facts | 3 |
| Notification order: every clock that applies, in order, each with an owner | 4 |
| DPDP pack: the Board note and the plain-language notice, with every required part | 6 |
| Tabletop decisions made and defended | 2 |
| Plain writing: facts kept apart from guesses | 3 |

- [ ] CERT-In filed with no problems left, and the update sent
- [ ] Notification order complete
- [ ] Board note and the customer notice written
- [ ] Tabletop played
- [ ] Press Hand in